## Setup

- **API key**: The first code cell loads your OpenRouter API key from a `.env` file. [OpenRouter](https://openrouter.ai) is a service that gives you one place to call many AI models. An API key is a secret code that identifies you to the service. The `.env` values replace any existing environment variables with the same name. If no key is found, the cell asks you to type it in. Never commit your key to Git.
- **Models**: The `MODEL` variable in `.env` chooses the model for the executor. If it is not set, the notebook uses `openai/gpt-6-luna`. `PLANNER_MODEL` is optional and chooses a different model for the planner. If it is not set, the planner uses `MODEL`. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models). They can change, so check there if the default stops working.
- **Search**: The search tool uses DuckDuckGo, so it needs no key.
- **Cost and network**: The notebook calls AI models and a search service over the internet. Each question makes several model calls, so it takes time and may use credit on your OpenRouter account, depending on the model you choose.

Your `.env` file needs these variable names (the values are your own secrets):

```
OPENROUTER_API_KEY=
MODEL=            # optional
PLANNER_MODEL=    # optional
```

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed
PLANNER_MODEL = os.getenv("PLANNER_MODEL", MODEL)  # optional: a different model for the planner

[Planning agents (LangChain blog)](https://blog.langchain.com/planning-agents/)

An article on planning agents, the broad approach this notebook's code uses.

# Plan-and-Solve (PS) Prompting

Plan-and-Solve (PS) prompting was introduced in a May 2023 paper, [Plan-and-Solve Prompting: Improving Zero-Shot Chain-of-Thought Reasoning by Large Language Models](https://arxiv.org/abs/2305.04091), by Lei Wang et al.

## Background: Zero-shot Chain-of-Thought

A **prompt** is the text you send to a language model. **Zero-shot** means you give the model no worked examples. **Chain-of-thought (CoT)** means asking the model to show its reasoning step by step. The simplest version adds the phrase "Let's think step by step" to the prompt. This is called zero-shot CoT.

Zero-shot CoT often helps, but the paper notes that it can still make calculation errors, skip steps, or misread the problem.

## The core idea

PS prompting splits the work into two stages:

1. **Plan**: the model breaks the task into smaller subtasks.
2. **Solve**: the model carries out those subtasks in order.

Instead of the vague "Let's think step by step", the paper uses a more structured instruction, such as "First, devise a plan. Then carry out the plan step by step." A variant called PS+ adds more detail, such as asking the model to identify relevant variables and compute intermediate results.

<figure>
  <img src="https://d3i71xaburhd42.cloudfront.net/62176de125738e3b95850d1227bac81fd646b78e/3-Figure2-1.png" alt="Example inputs and outputs of GPT-3 with zero-shot CoT, Plan-and-Solve prompting, and answer extraction prompting" style="width:80%">
  <figcaption>Example inputs and outputs of GPT-3 with (a) Zero-shot-CoT prompting, (b) Plan-and-Solve (PS) prompting, and (c) answer extraction prompting. Zero-shot-CoT may still produce wrong reasoning steps on complex problems. PS prompting first asks the model to devise a plan, then carry it out.</figcaption>
  <a href="https://ar5iv.labs.arxiv.org/html/2305.04091">Image Source</a>
</figure>

## What the authors report

The authors report that PS prompting beats zero-shot CoT on several arithmetic and reasoning datasets, and that it performs comparably to 8-shot CoT on arithmetic tasks. These are the authors' results, not ours. Check the paper for the exact numbers and datasets before you quote them.

## From prompting to agents

The paper describes one prompt that asks a single model to plan and then solve. The code in this notebook uses a related idea, with two separate roles:

- a **planner**, which writes the list of steps, and
- an **executor**, which carries out each step, using tools such as search and a calculator when it needs them.

## How the notebook's agent works

```mermaid
flowchart TD
    Q["User question"] --> P["Planner: writes the steps"]
    P --> S["Plan: numbered steps"]
    S --> E["Executor: runs one step, using tools if needed"]
    E --> R["Step result saved"]
    R -->|"more steps"| E
    R -->|"all steps done"| A["Final answer"]
```

If your notebook viewer does not render Mermaid diagrams, read the diagram as text: plan first, then run each step in order, then answer.

## Version note

The import below comes from `langchain_experimental`. The package's own warning says it is being sunset and is no longer actively maintained. The code is still useful for learning the planner and executor idea, but it may change or break in future versions. The warnings you may see when this cell runs (about deprecation, and about Pydantic on Python 3.14) come from that package and its dependencies. The warning links to [the GitHub issue on this](https://github.com/langchain-ai/langchain-experimental/issues/87).

In [2]:
from langchain_openrouter import ChatOpenRouter
from langchain_experimental.plan_and_execute import PlanAndExecute, load_chat_planner
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain.agents import create_agent

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/var/folders/vl/m6g01gp54cl4409b678xyrkm0000gn/T/ipykernel_26417/1847206468.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.plan_and_execute import PlanAndExecute, load_chat_planner
/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utili

# Define the Tools

A **tool** is a function that the agent is allowed to call. The language model does not run code itself. It asks for a tool by name and gives it an input, the program runs the tool, and the result goes back to the model. This pattern is called **tool calling**.

This notebook defines two tools:

- **Search** looks up current information with DuckDuckGo. It needs no API key. Search results can be out of date or wrong, so treat them as a starting point.
- **Calculator** does arithmetic.

Some details in the next cell are worth noticing:

- The `@tool("Search")` line turns a normal Python function into a tool. The docstring (the text in triple quotes) is what the model reads when it decides whether to use the tool, so the wording matters.
- If the search fails, the function returns an error message instead of raising an error. The agent sees the message and can carry on with the rest of the plan.
- The calculator does not use `eval()`. Python's `eval()` can run any code, which is risky when the input comes from a language model. This calculator reads the expression with the `ast` module and allows only numbers, the operators `+`, `-`, `*`, `/` and `**`, and the functions `abs` and `round`.

In [3]:
search_run = DuckDuckGoSearchRun()

@tool("Search")
def search_tool(query: str) -> str:
    """Useful for answering questions about current information."""
    try:
        return search_run.invoke(query)
    except Exception as e:  # a failed search is returned to the agent instead of stopping the run
        return f"Search failed: {e}"

# A small calculator. It evaluates arithmetic with the ast module, not eval(), so it runs only math.
import ast
import operator

_BIN_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
            ast.Div: operator.truediv, ast.Pow: operator.pow}
_UNARY_OPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}
_FUNCS = {"abs": abs, "round": round}

def _eval_node(node):
    if isinstance(node, ast.Expression):
        return _eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _BIN_OPS:
        return _BIN_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_eval_node(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _FUNCS:
        return _FUNCS[node.func.id](*[_eval_node(arg) for arg in node.args])
    raise ValueError("Only arithmetic is allowed")

@tool("Calculator")
def calculator_tool(expression: str) -> str:
    """Useful for when you need to do math. The input is an arithmetic expression, such as '2 * (3 + 4)' or 'abs(-4.6)'."""
    try:
        return str(_eval_node(ast.parse(expression, mode="eval")))
    except Exception as e:
        return f"Error: {e}"

agent_tools = [search_tool, calculator_tool]

# Create the Planner and Executor Agents

### Implement the planner

The **planner** is a language model with one job: write a plan. The plan is a numbered list of steps. The planner does not answer the question itself. `load_chat_planner` creates a planner with a built-in instruction that asks for a numbered list.

A **parser** turns the planner's reply into a list of steps. The notebook makes one small change. Sometimes the planner adds the marker `<END_OF_PLAN>` at the end of its reply. `CleanPlanParser` cuts the reply at that marker, so the marker and anything after it do not become a step.

The next code cell prints the exact instruction the planner receives. Read it to see what the planner is asked to do. Changing that instruction would change how plans are written, which is an optional experiment.

In [4]:
planner_model = ChatOpenRouter(model=PLANNER_MODEL, temperature=0)

planner = load_chat_planner(planner_model)

# The planner can write <END_OF_PLAN> into the last step. Cut the parsed plan at that marker.
from langchain_experimental.plan_and_execute.planners.chat_planner import PlanningOutputParser

class CleanPlanParser(PlanningOutputParser):
    def parse(self, text: str):
        return super().parse(text.split("<END_OF_PLAN>")[0].rstrip())

planner.output_parser = CleanPlanParser()

In [5]:
planner.llm_chain.prompt.messages[0].content

"Let's first understand the problem and devise a plan to solve the problem. Please output the plan starting with the header 'Plan:' and then followed by a numbered list of steps. Please make the plan the minimum number of steps required to accurately complete the task. If the task is a question, the final step should almost always be 'Given the above steps taken, please respond to the users original question'. At the end of your plan, say '<END_OF_PLAN>'"

### Assemble the agent

The **executor** carries out one step at a time. It is built with `create_agent`, which gives a language model a list of tools and runs a loop: the model decides whether to call a tool, the tool runs, and the model reads the result before it replies. The `system_prompt` tells the executor to finish the current step and reply with that step's result only.

`PlanAndExecute` expects its executor to have a `step()` method, which `create_agent` does not provide. `AgentStepExecutor` is a small adapter. It passes each step to the agent, along with the results of earlier steps, and returns the reply in the form `PlanAndExecute` expects.

Both models are created with `temperature=0`. Temperature controls how random a model's word choices are. Lower values give more predictable output, but they do not guarantee the same answer every time, and live search results can change between runs.

In [6]:
executor_model = ChatOpenRouter(model=MODEL, temperature=0)

# The executor is a create_agent agent. It uses native tool calling, so it does not need a JSON format.
# PlanAndExecute calls executor.step(), so this small wrapper adapts the agent to that interface.
from typing import Any
from langchain_core.messages import HumanMessage
from langchain_experimental.plan_and_execute.executors.base import BaseExecutor
from langchain_experimental.plan_and_execute.schema import ListStepContainer, StepResponse

class AgentStepExecutor(BaseExecutor):
    """Runs one plan step with a create_agent agent."""
    agent: Any

    def _step_message(self, inputs: dict) -> HumanMessage:
        previous = "\n".join(
            f"- {step.value}: {response.response}"
            for step, response in inputs["previous_steps"].get_steps()
        )
        text = f"Previous steps:\n{previous or 'None'}\n\nCurrent objective: {inputs['current_step'].value}"
        return HumanMessage(content=text)

    def step(self, inputs: dict, callbacks=None, **kwargs: Any) -> StepResponse:
        result = self.agent.invoke({"messages": [self._step_message(inputs)]}, config={"callbacks": callbacks})
        return StepResponse(response=result["messages"][-1].content)

    async def astep(self, inputs: dict, callbacks=None, **kwargs: Any) -> StepResponse:
        result = await self.agent.ainvoke({"messages": [self._step_message(inputs)]}, config={"callbacks": callbacks})
        return StepResponse(response=result["messages"][-1].content)

agent_executor = create_agent(
    executor_model,
    agent_tools,
    system_prompt="Complete the current objective. Use the tools when you need current information or math. Reply with the result of this step only.",
)
executor = AgentStepExecutor(agent=agent_executor)

agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)

In [7]:
agent.step_container = ListStepContainer()  # start each question with an empty step history
agent.invoke("What is the absolute value of the current temperature in Winnipeg, MB?")



> Entering new PlanAndExecute chain...
steps=[Step(value='Check a reliable weather website or app for the current temperature in Winnipeg, MB.'), Step(value="Determine the absolute value of the current temperature (if it's negative, convert it to positive)."), Step(value="Provide the absolute value of the current temperature.\n\nGiven the above steps taken, please respond to the user's original question.")]*****

Step: Check a reliable weather website or app for the current temperature in Winnipeg, MB.

Response: The current temperature in Winnipeg, MB is 43°F.*****

Step: Determine the absolute value of the current temperature (if it's negative, convert it to positive).

Response: The absolute value of the current temperature in Winnipeg, MB is 43.*****

Step: Provide the absolute value of the current temperature.

Given the above steps taken, please respond to the user's original question.

Response: The absolute value of the current temperature in Winnipeg, MB is 43.
> Finished ch

{'input': 'What is the absolute value of the current temperature in Winnipeg, MB?',
 'output': 'The absolute value of the current temperature in Winnipeg, MB is 43.'}

In [8]:
agent.step_container = ListStepContainer()  # start each question with an empty step history
agent.invoke("When is the next webinar on Direct Policy Optimization from DeepLearning.ai?")



> Entering new PlanAndExecute chain...
steps=[Step(value='Check the official DeepLearning.ai website or their events page for upcoming webinars.'), Step(value='Look for the specific webinar on Direct Policy Optimization and note the date and time.'), Step(value="If the information is not available on the website, check DeepLearning.ai's social media channels for announcements regarding webinars."), Step(value="Given the above steps taken, please respond to the user's original question.")]*****

Step: Check the official DeepLearning.ai website or their events page for upcoming webinars.

Response: I couldn't find specific upcoming webinars listed on the DeepLearning.ai website or their events page. You may want to check their official site directly for the most current information on events and webinars.*****

Step: Look for the specific webinar on Direct Policy Optimization and note the date and time.

Response: I couldn't find specific information regarding a webinar on Direct Polic

{'input': 'When is the next webinar on Direct Policy Optimization from DeepLearning.ai?',
 'output': "I couldn't find specific information regarding upcoming webinars on Direct Policy Optimization from DeepLearning.ai, either on their official website or social media channels. For the most current updates, I recommend checking their official Twitter account [@DeepLearningAI](https://twitter.com/DeepLearningAI) and their Facebook page directly."}

## Reading the two runs above

Both runs use `verbose=True`, so the output shows the plan the planner wrote, the response to each step, and the final answer. It does not show the tool calls, so the saved output cannot tell us whether the search tool ran.

**The Winnipeg run.** The planner wrote three steps for a question with one fact in it. The answer is `43`, which matches the number in the step responses, but the unit (°F, from the search step) was dropped from the final answer. Ask yourself where the `43` came from: a search result, or the model's memory?

**The webinar run.** The first two steps could not find the webinar. The planner had already written a fallback step (check social media), and the executor carried it out. The final answer lists social media channels and says what they usually post. It also mentions a search error for LinkedIn. Ask which of these details came from a tool result and which the model filled in. Then open the links yourself.

## Limitations and responsible use

- **Cost and speed.** Every step is a model call, and the planner and executor both call the model. More steps mean more cost and waiting.
- **Plans can be wrong.** The planner writes the whole plan before it sees any results. A bad plan leads to bad steps, and the executor will still try to carry them out.
- **Tools can be stale or wrong.** Search results depend on what the search engine returns on the day you run the notebook. Tools can also fail, as the webinar run shows.
- **Give agents only the tools they need.** This calculator only does arithmetic. A tool that could send messages or delete files would make a model's mistakes much more costly.
- **Check the sources.** Ask an agent for its sources and open them yourself. A confident answer is not evidence that it is correct.